In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from scipy.spatial import KDTree
from sklearn.base import BaseEstimator, TransformerMixin
from collections import defaultdict
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectFromModel
import warnings
warnings.filterwarnings('ignore')


In [9]:
# ---------------------- 配置参数 ----------------------
file_path = "D:/pythonai/midterm/"  # 数据文件路径

train_price = pd.read_csv(f"{file_path}ruc_Class25Q2_train_price.csv")
test_price = pd.read_csv(f"{file_path}ruc_Class25Q2_test_price.csv")
train_rent = pd.read_csv(f"{file_path}ruc_Class25Q2_train_rent.csv")
test_rent = pd.read_csv(f"{file_path}ruc_Class25Q2_test_rent.csv")
        
        # 备份测试集ID
test_price_ids = test_price[['ID']].copy()
test_rent_ids = test_rent[['ID']].copy()
        

print(f"房价训练集: {train_price.shape}, 房价测试集: {test_price.shape}")
print(f"租金训练集: {train_rent.shape}, 租金测试集: {test_rent.shape}")
        
       

房价训练集: (103871, 55), 房价测试集: (34017, 55)
租金训练集: (98899, 46), 租金测试集: (9773, 46)


In [10]:

def handle_rent_payment(df):
    """处理租金付款方式：计算月租金"""
    df_clean = df.copy()
    if '付款方式' in df_clean.columns and 'Price' in df_clean.columns:
        # 映射付款方式到月数
        payment_map = {
            '月付': 1,
            '双月付': 2,
            '季付': 3,
            '半年付': 6,
            '年付': 12
        }
        
        # 填充付款方式缺失值为季付
        df_clean['付款方式'] = df_clean['付款方式'].fillna('季付')
        
        # 转换付款方式为月数，未知的按季付处理
        df_clean['付款月数'] = df_clean['付款方式'].map(payment_map).fillna(3)
        
        # 计算月租金
        df_clean['月租金'] = df_clean['Price'] / df_clean['付款月数']
        
        # 替换原Price为月租金
        df_clean['Price'] = df_clean['月租金']
        
        # 删除临时列
        df_clean = df_clean.drop(columns=['付款方式', '付款月数', '月租金'])
    
    return df_clean

In [11]:
def parse_chinese_number(s):
    """解析中文数字为阿拉伯数字"""
    if not isinstance(s, str) or s.strip() == '':
        return np.nan
    s = s.strip()
    cn_num = {'零':0, '一':1, '二':2, '两':2, '三':3, '四':4, '五':5, '六':6, '七':7, '八':8, '九':9,
              '十':10, '百':100, '千':1000, '万':10000}
    
    # 处理纯数字字符串
    if re.match(r'^\d+$', s):
        return float(s)
    
    total = 0
    current = 0
    
    for ch in s:
        if ch in cn_num:
            num = cn_num[ch]
            if num >= 10:  # 十、百、千、万
                if current == 0:
                    current = 1
                total += current * num
                current = 0
            else:  # 零-九
                current += num
    total += current
    return total if total != 0 else (current if current != 0 else np.nan)


In [12]:
class LadderRatioExtractor(BaseEstimator, TransformerMixin):
    """提取梯户比例的转换器"""
    def __init__(self, col='梯户比例'):
        self.col = col
        
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        
        def parse_ratio(v):
            if pd.isna(v) or str(v).strip() == '':
                return 0.0
            s = str(v).strip()
            # 提取梯数和户数（支持中文数字和阿拉伯数字）
            ladders = re.findall(r'([一二三四五六七八九十百零两\d]+)梯', s)
            units = re.findall(r'([一二三四五六七八九十百零两\d]+)户', s)
            
            if ladders and units:
                a = parse_chinese_number(ladders[0])
                b = parse_chinese_number(units[0])
                if not np.isnan(a) and not np.isnan(b) and b != 0:
                    return round(a / b, 3)
            return 0.0
        
        X[self.col] = X[self.col].apply(parse_ratio)
        return X

In [13]:
def extract_year(s):
    """从字符串中提取四位数年份（避免datetime转换错误）"""
    if pd.isna(s) or str(s).strip() in ['', '0', '无']:
        return np.nan
    # 提取四位数年份（支持格式：2020年、2020/3/26、3018-03-26等）
    year_match = re.findall(r'(\d{4})', str(s))
    if year_match:
        year = int(year_match[0])
        # 过滤明显不合理的年份（1900-2025之间，超出则视为无效）
        return year if 1900 <= year <= 2025 else np.nan
    return np.nan  # 无有效年份


In [14]:
def extract_year_month(s):
    """从时间字符串中提取年和月（返回元组）"""
    if pd.isna(s):
        return (np.nan, np.nan)
    try:
        dt = pd.to_datetime(s, errors='coerce')
        return (dt.year, dt.month)
    except:
        return (np.nan, np.nan)



    

In [15]:
def extract_numeric_or_range(s):
    """
    提取数字或区间的中位数（适用于物业费、燃气费、供热费）
    示例："1.3-1.65元/月/㎡" → (1.3+1.65)/2=1.475；"2元" → 2
    """
    if pd.isna(s):
        return np.nan
    s = str(s).strip()
    # 提取所有数字（支持小数）
    nums = re.findall(r'(\d+\.?\d*)', s)
    if not nums:
        return np.nan
    # 转换为浮点数
    nums = [float(num) for num in nums]
    # 区间取中位数，单值直接返回
    return np.median(nums) if len(nums) >= 2 else nums[0]


In [16]:
def process_green_rate(s):
    """处理绿化率：提取数字并除以100"""
    if pd.isna(s):
        return np.nan
    nums = re.findall(r'(\d+\.?\d*)', str(s))
    if nums:
        rate = float(nums[0])
        # 处理百分比（如"30%"或"30"）
        return rate / 100 if rate > 1 else rate
    return np.nan

In [17]:
def process_parking_fee(s):
    """处理停车费用：“暂无”→0；有数字→提取数字；其他→缺失"""
    if pd.isna(s):
        return np.nan
    s = str(s).strip()
    # “暂无”转换为0
    if '暂无' in s:
        return 0.0
    # 提取数字
    nums = re.findall(r'(\d+\.?\d*)', s)
    if not nums:
        return np.nan  # 无数字视为缺失，后续用板块中位数填充
    return float(nums[0])
    

In [18]:
def extract_numeric(s):
    """提取字符串中的数字（适用于面积、价格等带单位的列）"""
    if pd.isna(s):
        return np.nan
    # 提取所有数字（支持小数）
    nums = re.findall(r'(\d+\.?\d*)', str(s))
    return float(nums[0]) if nums else np.nan  # 返回第一个数字


In [19]:
def parse_house_type(s):
    """解析户型：提取卧室数、总空间数、卧室占比（适用于售房“房屋户型”和租房“户型”）"""
    if pd.isna(s):
        s = '1室'  # 缺失值默认1室
    # 提取卧室数（“几室”）
    room = re.findall(r'(\d+)(?:室|房间)', s)
    room = int(room[0]) if room else 0
    # 提取总空间数（所有数字之和）
    total = sum(map(int, re.findall(r'\d+', s))) if re.findall(r'\d+', s) else 0
    ratio = room / total if total != 0 else 0
    return pd.Series([room, total, ratio])


In [20]:
def direction_dummies(df, col_name):
    """生成朝向哑变量（适用于售房“房屋朝向”和租房“朝向”）"""
    # 缺失值用众数填充
    df[col_name] = df[col_name].fillna(df[col_name].mode()[0])
    # 提取东南西北
    directions = ['东', '南', '西', '北']
    for dir in directions:
        df[f'朝向_{dir}'] = df[col_name].apply(lambda x: 1 if dir in str(x) else 0)
    return df.drop(columns=col_name)

In [21]:
def fill_by_group(df, col, group_col, stat='median'):
    """
    按分组填充缺失值
    stat: 'median'（中位数）或 'mode'（众数）
    """
    # 复制原始数据避免警告
    df_copy = df.copy()
    
    if stat == 'median':
        # 按分组计算中位数
        group_stat = df_copy.groupby(group_col)[col].transform('median')
        # 用全局中位数填充分组中位数仍缺失的值
        group_stat = group_stat.fillna(df_copy[col].median())
    elif stat == 'mode':
        # 按分组计算众数（取第一个众数）
        def get_mode(x):
            # 对每个分组计算众数，取第一个非空众数
            mode_vals = x.mode()
            return mode_vals.iloc[0] if not mode_vals.empty else np.nan
        
        group_stat = df_copy.groupby(group_col)[col].transform(get_mode)
        # 用全局众数填充分组众数仍缺失的值
        global_mode = df_copy[col].mode()
        global_mode_val = global_mode.iloc[0] if not global_mode.empty else '未知'
        group_stat = group_stat.fillna(global_mode_val)
    else:
        raise ValueError("stat参数只能是'median'或'mode'")
    
    return group_stat
    

In [22]:
def fill_plate_and_ring_by_latlon(df):
    """同时填充板块和环线的缺失值"""
    # 先填充板块
    df['板块'] = fill_by_latlon(df, '板块')
    # 再填充环线
    df['环线'] = fill_by_latlon(df, '环线')
    return df

In [23]:
def text_keyword_score(text, keywords):
    """文本关键词得分（适用于配套/交通）"""
    if pd.isna(text):
        return 0
    return sum(1 for kw in keywords if kw in str(text))


In [24]:
def fill_by_latlon(df,column_name):
    """用经纬度匹配最近的板块填充缺失值"""
    # 提取有目标列值和经纬度的样本构建KDTree
    known = df.dropna(subset=[column_name, 'lat', 'lon'])[['lat', 'lon', column_name]]
    if len(known) == 0:
        # 无参考数据则填充默认值
        default_value = '未知板块' if column_name == '板块' else '未知环线'
        return df[column_name].fillna(default_value)
    
    # 构建KDTree
    tree = KDTree(known[['lat', 'lon']].values)
    
    # 找出目标列缺失的样本
    missing = df[df[column_name].isna()][['lat', 'lon']]
    if len(missing) == 0:
        return df[column_name]  # 没有缺失值则直接返回
    
    # 查找最近邻（k=1）
    distances, indices = tree.query(missing.values, k=1)
    
    # 填充最近邻的目标列值
    df.loc[df[column_name].isna(), column_name] = known.iloc[indices.flatten()][column_name].values
    return df[column_name]


In [25]:
def final_missing_fill(df):
    """最终缺失值填充"""
    missing_cols = df.columns[df.isnull().any()].tolist()
    if missing_cols:
        print(f"检测到 {len(missing_cols)} 列存在缺失值，进行最终填充...")
        for col in missing_cols:
            if pd.api.types.is_numeric_dtype(df[col]):
                median_val = df[col].median()
                df[col] = df[col].fillna(median_val)
                print(f"  数值列 {col} 用中位数 {median_val:.2f} 填充")
            else:
                mode_val = df[col].mode()[0] if not df[col].mode().empty else '未知'
                df[col] = df[col].fillna(mode_val)
                print(f"  类别列 {col} 用众数 {mode_val} 填充")
    
    remaining_missing = df.isnull().sum().sum()
    if remaining_missing == 0:
        print("所有缺失值已成功填充")
    else:
        print(f"警告：仍存在 {remaining_missing} 个缺失值未处理")
    return df

In [26]:
def clean_data(df, data_type):
    """
    清洗数据主函数
    data_type: 'price'（售房）或 'rent'（租房）
    返回：清洗后的DataFrame
    """
    # 确保输入是DataFrame的副本，避免修改原数据
    df = df.copy()
    
    df.columns = [col.replace(' ', '') for col in df.columns]  # 移除列名中的所有空格
    # ---------------------- 1. 列名映射（统一处理逻辑） ----------------------
    col_mapping = {
        'price': {  # 售房列名映射
            '房屋户型': '户型',
            '房屋朝向': '朝向',
            '所在楼层': '楼层',
            '配备电梯': '电梯',
            '建筑结构_comm':'建筑结构',
            '供水': '用水',
            '供电': '用电',
            '供暖': '采暖'
        },
        'rent': {  # 租房列名映射
            '户型': '户型',
            '朝向': '朝向',
            '楼层': '楼层',
            '电梯': '电梯',
            '环线位置':'环线',
            '供暖': '采暖'
        }
    }
    # 重命名列名，统一为处理逻辑中的名称
    df = df.rename(columns=col_mapping.get(data_type, {}))
    # ---------------------- 2. 删除无关列 ----------------------
    drop_cols = {
        'price': [ '区域', '套内面积','抵押信息', '房屋用途','房屋优势', '核心卖点', 
                 '户型介绍', '年份', '开发商','物业公司','区县', '板块_comm', '环线位置', '物业办公电话', 
                 '产权描述', '房屋总数', '楼栋总数','coord_x', 'coord_y', '客户反馈'],
        'rent': ['租赁方式','装修', '租期', '车位', '客户反馈', '供水', '供电', '采暖', '年份', 
                '区县', '房屋总数','交易时间','开发商', '物业公司','楼栋总数', '物业办公电话', '产权描述', 'coord_x', 'coord_y']
    }
    df = df.drop(columns=drop_cols.get(data_type, []), errors='ignore')
    # ---------------------- 3. 面积提取 ----------------------
    if '建筑面积' in df.columns:
        df['面积'] = df['建筑面积'].apply(extract_numeric)
        df = df.drop(columns='建筑面积')
    elif '面积' in df.columns:
        df['面积'] = df['面积'].apply(extract_numeric)
     # ---------------------- 4. 梯户比例处理 ----------------------
    if '梯户比例' in df.columns:
        try:
            ladder_extractor = LadderRatioExtractor(col='梯户比例')
            df = ladder_extractor.transform(df)
        except Exception as e:
            print(f"处理梯户比例时出错: {e}")
            df = df.drop(columns=['梯户比例'], errors='ignore')
   
    # 5.6 板块、环线缺失值处理（用经纬度匹配）
    if 'lat' in df.columns and 'lon' in df.columns:  # 确保经纬度存在
        try:
            # 调用函数，获取填充后的板块和环线
            filled_data = fill_plate_and_ring_by_latlon(df)
            
            # 分别更新板块和环线列（只更新存在的列）
            if '板块' in df.columns and '板块' in filled_data.columns:
                df['板块'] = filled_data['板块']
            if '环线' in df.columns and '环线' in filled_data.columns:
                df['环线'] = filled_data['环线']
        except Exception as e:
            print(f"用经纬度匹配板块和环线时出错: {e}")

     # 11. 环线处理（转化为1-11编码）
    if '环线' in df.columns:
        circle_order = {
            '内环内': 1, '二环内': 2, '二至三环': 3, '三至四环': 4,
            '四至五环': 5, '五至六环': 6, '六环外': 7,
            '内环至中环': 8, '中环至外环': 9, '内环至外环': 10,
            '外环外': 11
        }
        df['环线_encoded'] = df['环线'].map(circle_order)
        df = df.drop(columns='环线')
    # ---------------------- 5. 共性列处理 ----------------------
    # 5.1 物业费、燃气费、供热费：提取数字或区间中位数
    for col in ['物业费', '燃气费', '供热费','停车位']:
        if col in df.columns:
            try:
                df[col] = df[col].apply(extract_numeric_or_range) 
                if '板块' in df.columns and '户型' in df.columns:
                   df[col] = fill_by_group(df, col, ['板块', '户型'], stat='median')
            except Exception as e:
                print(f"处理{col}时出错: {e}")
                df = df.drop(columns=[col], errors='ignore')
    
    
    # 5.2 绿化率：提取数字并除以100
    if '绿化率' in df.columns:
        try:
            df['绿化率'] = df['绿化率'].apply(process_green_rate)
            # 用同板块同户型的中位数填充
            if '板块' in df.columns and '户型' in df.columns:
                df['绿化率'] = fill_by_group(df, '绿化率', ['板块', '户型'], stat='median')
        except Exception as e:
            print(f"处理绿化率时出错: {e}")
            df = df.drop(columns=['绿化率'], errors='ignore')
    
    # 5.3 停车费用：“暂无”→0，提取数字，再用板块中位数填充缺失
    if '停车费用' in df.columns:
        try:
            df['停车费用'] = df['停车费用'].apply(process_parking_fee)
            if '板块' in df.columns and '户型' in df.columns:
                df['停车费用'] = fill_by_group(df, '停车费用', ['板块', '户型'], stat='median')
        except Exception as e:
            print(f"处理停车费用时出错: {e}")
            df = df.drop(columns=['停车费用'], errors='ignore')
    
    # 5.4 户型处理（售房“房屋户型”/租房“户型”）
    if '户型' in df.columns:
        try:
            df[['卧室数', '总空间数', '卧室占比']] = df['户型'].apply(parse_house_type)
            df['户型'] = fill_by_group(df, '户型', ['板块'], stat='mode')
        except Exception as e:
            print(f"处理户型时出错: {e}")
            df = df.drop(columns=['户型'], errors='ignore')
    # 5.5 朝向处理（售房“房屋朝向”/租房“朝向”）
    if '朝向' in df.columns:
        try:
            df = direction_dummies(df, '朝向')
        except Exception as e:
            print(f"处理朝向时出错: {e}")
            df = df.drop(columns=['朝向'], errors='ignore')
    
    # 5.2 物业费处理（列名"物业费"）
    if '物业费' in df.columns:
        try:
            df['物业费'] = df['物业费'].apply(extract_numeric_or_range)  # 提取区间中位数
            if '板块' in df.columns and '户型' in df.columns:
                df['物业费'] = fill_by_group(df, '物业费', ['板块', '户型'], stat='median')
        except Exception as e:
            print(f"处理物业费时出错: {e}")
            df = df.drop(columns=['物业费'], errors='ignore')
    
     # 12. 楼层处理
    if '楼层' in df.columns:
        try:
            def parse_floor(s):
                s = str(s)
                if '/' in s:
                    curr = re.findall(r'(\d+)/', s)
                    total = re.findall(r'/(\d+)层', s)
                    curr = int(curr[0]) if curr else np.nan
                    total = int(total[0]) if total else np.nan
                else:
                    total = re.findall(r'共(\d+)层', s)
                    total = int(total[0]) if total else np.nan
                    curr = np.nan
                return pd.Series([curr, total])
            
            df[['当前层数', '总层数']] = df['楼层'].apply(parse_floor)
            # 填充总层数缺失值
            if '板块' in df.columns:
                df['总层数'] = fill_by_group(df, '总层数', ['板块'], stat='median')
            
            # 划分楼层类型并编码
            def floor_type(row):
                total = row['总层数']
                curr = row['当前层数']
                if pd.isna(curr):
                    s = str(row['楼层'])
                    if '低' in s: return '低楼层'
                    elif '中' in s: return '中楼层'
                    elif '高' in s: return '高楼层'
                    elif '顶' in s: return '顶层'
                    elif '底' in s: return '底层'
                    elif '地下' in s: return '地下室'
                    else: return '中楼层'
                else:
                    if curr == 1: return '底层'
                    if curr == total: return '顶层'
                    ratio = curr / total
                    if ratio <= 1/3: return '低楼层'
                    elif ratio <= 2/3: return '中楼层'
                    else: return '高楼层'
            
            df['楼层类型'] = df.apply(floor_type, axis=1)
            floor_order = {'底层':1, '低楼层':2, '中楼层':3, '高楼层':4, '顶层':5, '地下室':6}
            df['楼层类型_encoded'] = df['楼层类型'].map(floor_order)
            df = df.drop(columns=['楼层'])
        except Exception as e:
            print(f"处理楼层时出错: {e}")
            df = df.drop(columns=['楼层'], errors='ignore')
    
    # ---------------------- 处理所有类别列（转换为哑变量或舍弃） ----------------------
    # 定义需要处理的所有类别列
    categorical_cols = ['装修情况', '交易权属', '产权所属', '物业类别', '建筑结构', '用水', '用电']
    
    for col in categorical_cols:
        # 跳过不存在的列
        if col not in df.columns:
            print(f"警告：列 '{col}' 不存在，已跳过")
            continue
        
        print(f"处理列：{col}")
        
        try:
            # 1. 确保列是字符串类型，避免数据类型问题
            df[col] = df[col].astype(str).replace('nan', '未知')
            
            # 2. 填充缺失值（使用众数）
            mode_series = df[col].mode()
            if not mode_series.empty:
                fill_value = mode_series.iloc[0]  # 用位置索引获取众数
                missing_count = df[col].isna().sum()
                df[col] = df[col].fillna(fill_value)
                print(f"  填充缺失值：{missing_count} 个，使用众数 '{fill_value}'")
            else:
                df[col] = df[col].fillna('未知')
                print(f"  列中无有效数据，用'未知'填充")
            
            # 3. 安全获取类别数量
            unique_count = df[col].nunique()
            
            # 如果是Series，取第一个值
            if isinstance(unique_count, pd.Series):
                if not unique_count.empty:
                    unique_count = unique_count.iloc[0]
                else:
                    unique_count = 0
            
            # 转换为整数
            unique_count = int(unique_count)
            print(f"  类别数量：{unique_count}")
            
            # 4. 转换为哑变量或舍弃
            if unique_count <= 10 and unique_count > 0:
                dummies = pd.get_dummies(df[col], prefix=col, drop_first=True)
                df = pd.concat([df.drop(columns=[col]), dummies], axis=1)
                print(f"  已转换为哑变量，新增 {len(dummies.columns)} 列")
            else:
                df = df.drop(columns=[col])
                print(f"  类别数量过多（{unique_count}个）或无效，已舍弃该列")
        except Exception as e:
            print(f"  处理{col}时出错：{str(e)}，已舍弃该列")
            df = df.drop(columns=[col], errors='ignore')
    
    # 14. 配套设施处理（用、分割）
    if '配套设施' in df.columns:
        try:
            df['配套设施'] = df['配套设施'].fillna('')
            # 按、分割并去重
            df['配套_list'] = df['配套设施'].apply(
                lambda x: list(set(re.split(r'[、,，]', str(x)))) if x else []
            )
            # 提取高频特征
            all_features = []
            for lst in df['配套_list']:
                all_features.extend(lst)
            top_features = [f for f, _ in pd.Series(all_features).value_counts().head(50).items() if f]
            # 生成哑变量
            for f in top_features:
                df[f'配套_{f}'] = df['配套_list'].apply(lambda x: 1 if f in x else 0)
            df = df.drop(columns=['配套设施', '配套_list'])
        except Exception as e:
            print(f"处理配套设施时出错: {e}")
            df = df.drop(columns=['配套设施'], errors='ignore')
    
    # 5.8 电梯处理
    if '电梯' in df.columns:
        try:
            if '总层数' in df.columns:
                # 用总层数推断缺失的电梯信息
                df['电梯'] = df.apply(
                    lambda row: '有' if row['总层数'] > 6 else '无' 
                    if pd.isna(row['电梯']) else row['电梯'], axis=1
                )
            else:
                if '板块' in df.columns:
                    df['电梯'] = fill_by_group(df, '电梯', ['板块'], stat='mode')
            df = pd.get_dummies(df, columns=['电梯'], drop_first=True)
        except Exception as e:
            print(f"处理电梯时出错: {e}")
            df = df.drop(columns=['电梯'], errors='ignore')
    
    
    # 5.12 供暖处理（秦岭淮河界：北纬33°）
    if '采暖' in df.columns and 'lat' in df.columns:
        try:
            df['采暖'] = df['采暖'].apply(split_slash_text)
            df['采暖'] = df.apply(
                lambda row: '集中采暖' if row['lat'] >= 33 else '自采暖' 
                if pd.isna(row['采暖']) else row['采暖'], axis=1
            )
            df = pd.get_dummies(df, columns=['采暖'], drop_first=True)
        except Exception as e:
            print(f"处理采暖时出错: {e}")
            df = df.drop(columns=['采暖'], errors='ignore')
    
    # 18. 售房特有处理
    if data_type == 'price':
       # 移除当前层数列（仅在售房数据中）
        if '当前层数' in df.columns:
             df = df.drop(columns=['当前层数'])
             print("售房数据已移除'当前层数'列")
        # 6.2 别墅类型
        if '别墅类型' in df.columns:
            try:
                df['别墅类型'] = df['别墅类型'].fillna('非别墅')
                df = pd.get_dummies(df, columns=['别墅类型'], drop_first=True)
            except Exception as e:
                print(f"处理别墅类型时出错: {e}")
                df = df.drop(columns=['别墅类型'], errors='ignore')
        
        # 交易时间处理
        if '交易时间' in df.columns:
            try:
                df[['交易年份', '交易月份']] = df['交易时间'].apply(
                    lambda x: pd.Series(extract_year_month(x))
                )
                if '板块' in df.columns:
                    df['交易年份'] = fill_by_group(df, '交易年份', ['板块'], stat='median')
                    df['交易月份'] = fill_by_group(df, '交易月份', ['板块'], stat='median')
                df = df.drop(columns='交易时间')
            except Exception as e:
                print(f"处理交易时间时出错: {e}")
                df = df.drop(columns=['交易时间'], errors='ignore')
        
        # 建筑年代处理
        if '建筑年代' in df.columns:
            try:
                df['建筑年代_year'] = df['建筑年代'].apply(extract_year)
                if '板块' in df.columns and '户型' in df.columns:
                    df['建筑年代_year'] = fill_by_group(df, '建筑年代_year', ['板块', '户型'], stat='median')
                df = df.drop(columns='建筑年代')
            except Exception as e:
                print(f"处理建筑年代时出错: {e}")
                df = df.drop(columns=['建筑年代'], errors='ignore')
        
        # 房屋年限计算
        if '交易年份' in df.columns and '建筑年代_year' in df.columns:
            try:
                df['上次交易_year'] = df.get('上次交易', pd.Series(np.nan)).apply(extract_year)
                if '板块' in df.columns:
                    df['上次交易_year'] = fill_by_group(df, '上次交易_year', ['板块'], stat='median')
                
                def calculate_year_limit(row):
                    if not pd.isna(row['上次交易_year']) and row['上次交易_year'] > 0:
                        return row['交易年份'] - row['上次交易_year']
                    else:
                        return row['交易年份'] - row['建筑年代_year']
                
                df['房屋年限'] = df.apply(calculate_year_limit, axis=1)
                df['房屋年限'] = df['房屋年限'].clip(lower=0, upper=100)
                
                df['年限分组'] = pd.cut(
                    df['房屋年限'], bins=[-np.inf, 2, 5, np.inf], labels=['未满两年', '满两年', '满五年']
                )
                df = pd.get_dummies(df, columns=['年限分组'], drop_first=True)
                df = df.drop(columns=['上次交易', '房屋年限'], errors='ignore')
            except Exception as e:
                print(f"处理房屋年限时出错: {e}")
        
        # 周边配套和交通出行得分
        if '周边配套' in df.columns:
            try:
                support_kw = ['学校', '医院', '商场', '超市', '公园', '银行']
                df['配套得分'] = df['周边配套'].apply(lambda x: text_keyword_score(x, support_kw))
                df = df.drop(columns='周边配套')
            except Exception as e:
                print(f"处理周边配套时出错: {e}")
                df = df.drop(columns=['周边配套'], errors='ignore')
        
        if '交通出行' in df.columns:
            try:
                traffic_kw = ['地铁', '公交', '站', '高速', '路', '桥']
                df['交通得分'] = df['交通出行'].apply(lambda x: text_keyword_score(x, traffic_kw))
                df = df.drop(columns='交通出行')
            except Exception as e:
                print(f"处理交通出行时出错: {e}")
                df = df.drop(columns=['交通出行'], errors='ignore')
    
    # 19. 租房特有处理
    if data_type == 'rent':
        try:
            df = handle_rent_payment(df)
            if '燃气' in df.columns and '配套设施' in df.columns:
                df['燃气'] = df.apply(
                    lambda row: '有' if '天然气' in str(row.get('配套设施', '')) 
                    else '无' if pd.isna(row['燃气']) else row['燃气'], axis=1
                )
                df = pd.get_dummies(df, columns=['燃气'], drop_first=True)
        except Exception as e:
            print(f"处理租房特有数据时出错: {e}")
    
    # 确保卧室数等特征存在
    if '卧室数' not in df.columns and '户型' in df.columns:
        try:
            df[['卧室数', '总空间数', '卧室占比']] = df['户型'].apply(parse_house_type)
        except Exception as e:
            print(f"提取卧室数等特征时出错: {e}")
    
    # 强制删除户型列（如果存在）
    if '户型' in df.columns:
        df = df.drop(columns='户型')
    
    # 最终缺失值填充
    try:
        df = final_missing_fill(df)
    except Exception as e:
        print(f"最终缺失值填充时出错: {e}")
    
    # 确保返回的是DataFrame
    if not isinstance(df, pd.DataFrame):
        print("警告：数据处理后不是DataFrame类型，返回空DataFrame")
        return pd.DataFrame()
    
    return df
    

In [27]:
# ---------------------- 执行清洗（处理4个文件） ----------------------
if __name__ == "__main__":
   
    train_price_clean = clean_data(train_price, data_type='price')
    test_price_clean = clean_data(test_price, data_type='price')
    train_rent_clean = clean_data(train_rent, data_type='rent')
    test_rent_clean = clean_data(test_rent, data_type='rent')
    # 保存清洗后的结果
    train_price_clean.to_csv(f"{file_path}train_price_clean.csv", index=False)
    test_price_clean.to_csv(f"{file_path}test_price_clean.csv", index=False)
    train_rent_clean.to_csv(f"{file_path}train_rent_clean.csv", index=False)
    test_rent_clean.to_csv(f"{file_path}test_rent_clean.csv", index=False)
    
    print("清洗完成！")
    print(f"售房训练集形状：{train_price_clean.shape}")
    print(f"售房训练集缺失值总数：{train_price_clean.isnull().sum().sum()}")

    print(f"售房测试集形状：{test_price_clean.shape}")
    print(f"售房测试集缺失值总数：{test_price_clean.isnull().sum().sum()}")

    print(f"租房训练集形状：{train_rent_clean.shape}")
    print(f"租房训练集缺失值总数：{train_rent_clean.isnull().sum().sum()}")

    print(f"租房测试集形状：{test_rent_clean.shape}")
    print(f"租房测试集缺失值总数：{test_rent_clean.isnull().sum().sum()}")

处理列：装修情况
  填充缺失值：0 个，使用众数 '精装'
  类别数量：5
  已转换为哑变量，新增 4 列
处理列：交易权属
  填充缺失值：0 个，使用众数 '商品房'
  类别数量：15
  类别数量过多（15个）或无效，已舍弃该列
处理列：产权所属
  填充缺失值：0 个，使用众数 '非共有'
  类别数量：2
  已转换为哑变量，新增 1 列
处理列：物业类别
  填充缺失值：0 个，使用众数 '未知'
  类别数量：231
  类别数量过多（231个）或无效，已舍弃该列
处理列：建筑结构
  填充缺失值：建筑结构    0
建筑结构    0
dtype: int64 个，使用众数 '建筑结构    钢混结构
建筑结构      未知
Name: 0, dtype: object'
  类别数量：8
  已转换为哑变量，新增 22 列
处理列：用水
  填充缺失值：0 个，使用众数 '民水'
  类别数量：4
  已转换为哑变量，新增 3 列
处理列：用电
  填充缺失值：0 个，使用众数 '民电'
  类别数量：4
  已转换为哑变量，新增 3 列
处理采暖时出错: name 'split_slash_text' is not defined
售房数据已移除'当前层数'列
检测到 1 列存在缺失值，进行最终填充...
  数值列 容积率 用中位数 2.50 填充
所有缺失值已成功填充
处理列：装修情况
  填充缺失值：0 个，使用众数 '精装'
  类别数量：5
  已转换为哑变量，新增 4 列
处理列：交易权属
  填充缺失值：0 个，使用众数 '商品房'
  类别数量：15
  类别数量过多（15个）或无效，已舍弃该列
处理列：产权所属
  填充缺失值：0 个，使用众数 '非共有'
  类别数量：2
  已转换为哑变量，新增 1 列
处理列：物业类别
  填充缺失值：0 个，使用众数 '普通住宅'
  类别数量：214
  类别数量过多（214个）或无效，已舍弃该列
处理列：建筑结构
  填充缺失值：建筑结构    0
建筑结构    0
dtype: int64 个，使用众数 '建筑结构    钢混结构
建筑结构      板楼
Name: 0, dtype: object'
  类别数量：8
  已转换为哑变量，新增 22 列
处理列：用水

In [28]:
print(train_price_clean.head())
print(train_price_clean.info())
print(train_rent_clean.head())
print(train_rent_clean.info())

   城市    板块        Price   梯户比例         lon        lat    绿化率   容积率    物业费  \
0   0   150  6194048.992  0.333  117.424278  40.975752  0.300  3.00  1.475   
1   0   299  4354153.263  0.500  117.389228  41.091295  0.300  1.73  0.650   
2   0   911  3321991.616  0.200  117.200934  40.747919  0.300  1.70  1.130   
3   0  1102  7895655.584  0.000  117.767308  41.228803  0.401  1.00  3.200   
4   0   295  1902960.295  0.182  117.334530  40.952530  0.600  1.58  5.150   

    燃气费  ...  别墅类型_联排  别墅类型_非别墅    交易年份  交易月份  建筑年代_year  上次交易_year  年限分组_满两年  \
0  2.61  ...    False      True  2024.0   8.0     1955.0     2014.0     False   
1  2.61  ...    False      True  2024.0   8.0     2003.0     2012.0     False   
2  2.61  ...    False      True  2024.0   7.0     2003.0     2016.0     False   
3  2.62  ...    False     False  2024.0   8.0     2015.0     2018.0     False   
4  2.62  ...    False      True  2024.0   8.0     2003.0     2017.0     False   

   年限分组_满五年  配套得分  交通得分  
0      True     3 

In [1]:
def handle_outliers(df, target, method='iqr'):
    """使用IQR或Z-score处理目标变量异常值"""
    if method == 'iqr':
        q1 = df[target].quantile(0.25)
        q3 = df[target].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        return df[(df[target] >= lower) & (df[target] <= upper)]
    else:  # z-score
        z = np.abs(stats.zscore(df[target]))
        return df[z <= 3]

In [ ]:
def filter_numeric_features(X_train, X_test):
    """筛选数值型特征"""
    numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
    numeric_cols = [col for col in numeric_cols if col != 'ID']  # 移除ID列
    print(f"保留的数值特征数量: {len(numeric_cols)}")
    return X_train[numeric_cols].copy(), X_test[numeric_cols].copy(), numeric_cols


def create_new_features(df, num_cols=None):
    """创建新特征"""
    df_new = df.copy()
    
    # 对数变换（排除目标变量）
    if num_cols is None:
        num_cols = df_new.select_dtypes(include=['float64', 'int64']).columns.tolist()
    
    for col in num_cols:
        if df_new[col].min() > 0 and col not in ['ID']:
            df_new[f'log_{col}'] = np.log1p(df_new[col])
    
    # 交互项
    if '面积' in df_new.columns and '卧室数' in df_new.columns:
        df_new['面积_卧室数交互'] = df_new['面积'] * df_new['卧室数']
    if '环线_encoded' in df_new.columns and '交通得分' in df_new.columns:
        df_new['环线_交通交互'] = df_new['环线_encoded'] * df_new['交通得分']
    
    return df_new

In [4]:
def handle_missing_values(X_train, X_test, strategy='median'):
    """处理缺失值，使用训练集的统计量填充"""
    imputer = SimpleImputer(strategy=strategy)
    X_train_imputed = imputer.fit_transform(X_train)
    X_test_imputed = imputer.transform(X_test)
    
    # 转换回DataFrame
    X_train_imputed = pd.DataFrame(
        X_train_imputed, columns=X_train.columns, index=X_train.index
    )
    X_test_imputed = pd.DataFrame(
        X_test_imputed, columns=X_test.columns, index=X_test.index
    )
    
    # 检查缺失值
    train_nan_count = X_train_imputed.isna().sum().sum()
    test_nan_count = X_test_imputed.isna().sum().sum()
    print(f"缺失值处理后 - 训练集剩余NaN: {train_nan_count}, 测试集剩余NaN: {test_nan_count}")
    
    return X_train_imputed, X_test_imputed

In [5]:
def feature_selection_lasso(X, y, alpha=0.1):
    """使用Lasso进行特征选择"""
    # 检查并处理NaN值
    nan_count = X.isna().sum().sum()
    if nan_count > 0:
        print(f"警告：输入数据中存在{nan_count}个NaN值，将使用中位数填充")
        imputer = SimpleImputer(strategy='median')
        X_imputed = imputer.fit_transform(X)
        X_imputed = pd.DataFrame(X_imputed, columns=X.columns)
    else:
        X_imputed = X.copy()
    
    lasso = Lasso(alpha=alpha, random_state=111, max_iter=10000)
    lasso.fit(X_imputed, y)
    
    # 选择非零系数特征
    selected_features = X.columns[lasso.coef_ != 0].tolist()
    print(f"Lasso特征选择后保留特征数：{len(selected_features)}")
    
    return selected_features

In [31]:
def train_evaluate_model(X_train, y_train, X_val, y_val, model_name, model, param_grid=None):
    """训练模型并评估性能"""
    # 超参数调优
    if param_grid:
        grid = GridSearchCV(
            model, param_grid, cv=6, scoring='neg_mean_absolute_error', n_jobs=-1
        )
        grid.fit(X_train, y_train)
        best_model = grid.best_estimator_
        print(f"{model_name} 最佳参数：{grid.best_params_}")
    else:
        best_model = model.fit(X_train, y_train)
    
    # 预测与评估
    y_train_pred = best_model.predict(X_train)
    y_val_pred = best_model.predict(X_val)
    
    metrics = {
        'in_sample_mae': mean_absolute_error(y_train, y_train_pred),
        'in_sample_rmse': np.sqrt(mean_squared_error(y_train, y_train_pred)),
        'out_sample_mae': mean_absolute_error(y_val, y_val_pred),
        'out_sample_rmse': np.sqrt(mean_squared_error(y_val, y_val_pred)),
        'cv_mae': -cross_val_score(best_model, X_train, y_train, cv=6, scoring='neg_mean_absolute_error').mean(),
        'cv_rmse': np.sqrt(-cross_val_score(best_model, X_train, y_train, cv=6, scoring='neg_mean_squared_error').mean())
    }
    # 打印结果
    print(f"\n{model_name} 性能：")
    print(f"样本内 MAE: {metrics['in_sample_mae']:.2f}, RMSE: {metrics['in_sample_rmse']:.2f}")
    print(f"样本外 MAE: {metrics['out_sample_mae']:.2f}, RMSE: {metrics['out_sample_rmse']:.2f}")
    print(f"6折交叉验证 MAE: {metrics['cv_mae']:.2f}, RMSE: {metrics['cv_rmse']:.2f}")
    
    return best_model, metrics

In [32]:
# ---------------------- 主处理函数 ----------------------
def process_and_predict(data_type, train_data, test_data, target_col, test_ids, file_path):
    """
    处理数据并生成所有模型的预测结果
    data_type: 'price' 或 'rent'
    """
    print("\n" + "="*50)
    print(f"开始处理{data_type}数据")
    print("="*50)
    
    # 1. 分离特征和目标变量
    drop_cols = [target_col, 'ID'] if 'ID' in train_data.columns else [target_col]
    X = train_data.drop(columns=drop_cols)
    y = train_data[target_col]
    X_test = test_data.drop(columns=['ID'])
    
    print(f"{data_type}训练集形状: {X.shape}")
    print(f"{data_type}测试集形状: {X_test.shape}")
    
    # 2. 筛选数值特征
    X_filtered, X_test_filtered, numeric_features = filter_numeric_features(X, X_test)
    
    # 3. 处理异常值
    train_combined = pd.concat([X_filtered, y], axis=1)
    train_no_outlier = handle_outliers(train_combined, target_col)
    print(f"{data_type}数据去除异常值后样本数: {len(train_no_outlier)} (原样本数: {len(train_combined)})")
    
    # 4. 特征工程
    X_train = train_no_outlier.drop(columns=[target_col])
    y_train = train_no_outlier[target_col]
    X_train_fe = create_new_features(X_train)
    X_test_fe = create_new_features(X_test_filtered)
    
    # 5. 特征选择
    selected_features = feature_selection_lasso(X_train_fe, y_train, alpha=0.01)
    
    # 6. 划分训练集和验证集
    X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
        X_train_fe[selected_features], y_train,
        test_size=0.2, random_state=111
    )
    print(f"训练集大小: {X_train_split.shape}, 验证集大小: {X_val_split.shape}")
    
    # 7. 特征标准化
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_split)
    X_val_scaled = scaler.transform(X_val_split)
    
    # 8. 定义模型
    models = {
        'OLS': (LinearRegression(), None),
        'Lasso': (Lasso(random_state=111, max_iter=10000), 
                 {'alpha': [0.001, 0.01, 0.1, 1, 10]}),
        'Ridge': (Ridge(random_state=111), 
                 {'alpha': [0.001, 0.01, 0.1, 1, 10, 100]}),
        'ElasticNet': (ElasticNet(random_state=111, max_iter=10000), 
                      {'alpha': [0.001, 0.01, 0.1], 'l1_ratio': [0.3, 0.5, 0.7]})
    }
    
    # 9. 训练所有模型并保存预测结果
    trained_models = {}
    model_metrics = {}
    predictions = {}
    
    for name, (model, param_grid) in models.items():
        print(f"\n训练 {name} 模型...")
        best_model, metrics = train_evaluate_model(
            X_train_scaled, y_train_split, X_val_scaled, y_val_split,
            name, model, param_grid
        )
        trained_models[name] = best_model
        model_metrics[name] = metrics
        
        # 生成测试集预测
        missing_feats = set(selected_features) - set(X_test_fe.columns)
        for feat in missing_feats:
            X_test_fe[feat] = 0
        
        X_test_final = X_test_fe[selected_features]
        X_test_scaled = scaler.transform(X_test_final)
        pred = best_model.predict(X_test_scaled)
        predictions[name] = pred
        
        # 保存单个模型预测结果
        submission = pd.DataFrame({
            'ID': test_ids['ID'].values,
            'Price': pred
        })
        submission.to_csv(f"{file_path}{data_type}_{name}_prediction.csv", index=False)
        print(f"{data_type}的{name}模型预测已保存至 {file_path}{data_type}_{name}_prediction.csv")
    
    # 10. 确定最佳模型
    best_model_name = min(model_metrics, key=lambda x: model_metrics[x]['out_sample_mae'])
    print(f"\n{data_type}最佳模型: {best_model_name}")
    
    return trained_models, model_metrics, best_model_name


In [34]:
if __name__ == "__main__":
    # 配置文件路径（根据实际情况修改）
    file_path = "D:/pythonai/midterm/"  # 数据文件路径
    
    # 加载清洗后的数据集
    train_price = pd.read_csv(f"{file_path}train_price_clean.csv")
    test_price = pd.read_csv(f"{file_path}test_price_clean.csv")
    train_rent = pd.read_csv(f"{file_path}train_rent_clean.csv")
    test_rent = pd.read_csv(f"{file_path}test_rent_clean.csv")
    
    # 提取测试集ID
    test_price_ids = test_price[['ID']].copy()
    test_rent_ids = test_rent[['ID']].copy()
    
    # 处理房价数据并生成预测
    price_models, price_metrics, best_price_model = process_and_predict(
        data_type='price',
        train_data=train_price,
        test_data=test_price,
        target_col='Price',
        test_ids=test_price_ids,
        file_path=file_path
    )
    
    # 处理租金数据并生成预测
    rent_models, rent_metrics, best_rent_model = process_and_predict(
        data_type='rent',
        train_data=train_rent,
        test_data=test_rent,
        target_col='Price',
        test_ids=test_rent_ids,
        file_path=file_path
    )
    
    # 生成性能报告
    print("\n" + "="*80)
    print("模型性能总结")
    print("="*80)
    
    # 房价模型性能
    print("\n房价模型性能:")
    print(f"{'模型':<15} {'样本内MAE':<12} {'样本外MAE':<12} {'交叉验证MAE':<15}")
    print("-" * 60)
    for name in price_metrics:
        metrics = price_metrics[name]
        print(f"{name:<15} {metrics['in_sample_mae']:<12.2f} {metrics['out_sample_mae']:<12.2f} {metrics['cv_mae']:<15.2f}")
    print(f"\n房价最佳模型: {best_price_model}")
    
    # 租金模型性能
    print("\n租金模型性能:")
    print(f"{'模型':<15} {'样本内MAE':<12} {'样本外MAE':<12} {'交叉验证MAE':<15}")
    print("-" * 60)
    for name in rent_metrics:
        metrics = rent_metrics[name]
        print(f"{name:<15} {metrics['in_sample_mae']:<12.2f} {metrics['out_sample_mae']:<12.2f} {metrics['cv_mae']:<15.2f}")
    print(f"\n租金最佳模型: {best_rent_model}")


开始处理price数据
price训练集形状: (103871, 70)
price测试集形状: (34017, 70)
保留的数值特征数量: 29
price数据去除异常值后样本数: 96048 (原样本数: 103871)
Lasso特征选择后保留特征数：47
训练集大小: (76838, 47), 验证集大小: (19210, 47)

训练 OLS 模型...

OLS 性能：
样本内 MAE: 572031.34, RMSE: 763043.91
样本外 MAE: 570653.47, RMSE: 759476.41
6折交叉验证 MAE: 572474.70, RMSE: 763739.03
price的OLS模型预测已保存至 D:/pythonai/midterm/price_OLS_prediction.csv

训练 Lasso 模型...
Lasso 最佳参数：{'alpha': 0.001}

Lasso 性能：
样本内 MAE: 573141.31, RMSE: 763718.10
样本外 MAE: 571465.52, RMSE: 760006.21
6折交叉验证 MAE: 573561.13, RMSE: 764381.81
price的Lasso模型预测已保存至 D:/pythonai/midterm/price_Lasso_prediction.csv

训练 Ridge 模型...
Ridge 最佳参数：{'alpha': 0.001}

Ridge 性能：
样本内 MAE: 572052.36, RMSE: 763044.18
样本外 MAE: 570678.70, RMSE: 759485.79
6折交叉验证 MAE: 572499.14, RMSE: 763738.23
price的Ridge模型预测已保存至 D:/pythonai/midterm/price_Ridge_prediction.csv

训练 ElasticNet 模型...
ElasticNet 最佳参数：{'alpha': 0.001, 'l1_ratio': 0.7}

ElasticNet 性能：
样本内 MAE: 573895.49, RMSE: 764699.55
样本外 MAE: 571812.57, RMSE: 760882.67
6折交叉验